# 6. Thứ tự FEFO
Dự đoán mỗi dòng kết quả biểu diễn điều gì.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'run_inventory.py').is_file())
sys.path.insert(0, str(root))
from inventory_lab.db import connect, DEFAULT_DB, LAST
from inventory_lab.sql_metrics import query
db = Path(os.environ.get('INVENTORY_LAB_DB', DEFAULT_DB))
assert db.is_file(), 'Chạy python src/prepare_data.py trong PowerShell'
def read(sql, params=None):
    with connect(db) as connection:
        return pd.read_sql_query(sql, connection, params=params)

result = read("SELECT lot_id,product_id,ROW_NUMBER() OVER(PARTITION BY product_id ORDER BY expires_on,received_on,lot_id) AS position FROM lots")
result.head()

Phá hòa bằng ngày nhập rồi mã lô. Phải lọc lô đủ điều kiện trước khi dùng cho xuất.

**Tự làm**

Thêm điều kiện hạn và tồn.

In [ ]:
assert result.groupby('product_id').position.min().eq(1).all()